# Validate the 1st order consensus against test data from Ciara

There isn't enough human-transcribed data to train against, but we can use it for validating the AI transcription models

This notebook documents the validation against the models fine-tuned against fake images.


In [1]:
import subprocess

# Define the checkpoints to be used for validation.
# Each checkpoint is associated with a model name, the path to the checkpoint, the batch size for processing, and the number of shards to split the workload into. Checkpoints are the models after fine-tuning against the fake images.
# Name, checkpoint path, batch size, no.of shards
# Checkpoints are the models after fine-tuning against the fake images.
MODEL_JOBS = [
    (
        "SmolVLM",
        "Daily_rainfall_sample/outputs/checkpoints/smolvlm2-20260601-154723/HuggingFaceTB--SmolVLM2-2.2B-Instruct",
        50,1,
    ),
    (
        "Granite4",
        "Daily_rainfall_sample/outputs/checkpoints/granite4-20260601-121821/ibm-granite--granite-vision-4.1-4b",
        50,1,
    ),
    (
        "Gemma3",
        "Daily_rainfall_sample/outputs/checkpoints/gemma3-20260601-121832/google--gemma-3-4b-it",
        50,1,
    ),
    (
        "Gemma4",
        "Daily_rainfall_sample/outputs/checkpoints/gemma4-20260601-121840/google--gemma-4-E4B-it",
        50,1,
    ),
    (
        "Ministral",
        "Daily_rainfall_sample/outputs/checkpoints/ministral-20260601-121858/mistralai--Mistral-Small-3.1-24B-Instruct-2503",
        15,1,
    ),
]


## 2. Run the extractions

The code cell below submits one extraction job for each of the five models.
It relies on the MODEL_JOBS dictionary defined in the cell above.

There are only 64 images in the test set, so these jobs will run in 15 mins or so (+queueing time).

In [ ]:

for model_name, checkpoint, batch_size, total_shards, in MODEL_JOBS:
    print(f"Submitting {model_name}...")
    subprocess.run(
        [
            "bash",
            "../scripts/aml_submit.sh",
            "--checkpoint",
            checkpoint,
            "--images-path",
            "test_data/from_Ciara/images",
            "--transcriptions-path",
            "documents/DailyRainfall_UK/test_data/from_Ciara/transcriptions_1",
            "--total-shards",
            str(total_shards),
            "--batch-size",
            str(batch_size),
            "--extraction-registry",
            "../outputs/extraction_registry.json",
            "extract",
        ],
        check=True,
    )

When the jobs have been submitted, the extraction registry will contain the run names we need to download and analyze the extractions.

Run Cell 6 to discover run names from the registry.
Then copy the printed `RUN_NAMES = [...]` block into Cell 7 so the run names are stored directly in this notebook.

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path
from posixpath import normpath

# Discover run names from extraction registry after submissions complete.
# This cell does NOT write external files. It prints a block to paste into Cell 7.


def _first_existing_path(candidates: list[Path]) -> Path:
    for p in candidates:
        if p.exists():
            return p
    return candidates[0]


REGISTRY_PATH = _first_existing_path(
    [Path("outputs/extraction_registry.json"), Path("../outputs/extraction_registry.json")]
 )
TARGET_IMAGES_PATH = "test_data/from_Ciara/images"


def _norm_rel(path_like: str) -> str:
    return normpath(path_like.replace("\\", "/")).lstrip("./")


def _parse_created_at(value: str) -> datetime:
    if not value:
        return datetime.min.replace(tzinfo=timezone.utc)
    try:
        return datetime.fromisoformat(value.replace("Z", "+00:00"))
    except ValueError:
        return datetime.min.replace(tzinfo=timezone.utc)


registry = json.loads(REGISTRY_PATH.read_text(encoding="utf-8"))
entries = registry.get("extractions", [])

RUN_NAMES = []
missing = []
target_images_norm = _norm_rel(TARGET_IMAGES_PATH)

for model_name, checkpoint, _batch_size, _total_shards in MODEL_JOBS:
    candidates = [
        e
        for e in entries
        if _norm_rel(str(e.get("images_path", ""))) == target_images_norm
        and str(e.get("checkpoint_path", "")) == checkpoint
        and e.get("run_name")
    ]
    if not candidates:
        missing.append(model_name)
        continue
    best = max(
        candidates,
        key=lambda e: _parse_created_at(str(e.get("created_at", ""))),
    )
    run_name = str(best["run_name"] )
    RUN_NAMES.append(run_name)
    print(f"{model_name}: {run_name}")

if missing:
    raise RuntimeError(
        "Missing extraction runs in registry for: "
        + ", ".join(missing)
        + ". Run the extraction submission cell first, then re-run this cell."
    )

EXTRACTION_DIRS = [f"../outputs/extractions/{r}" for r in RUN_NAMES]

print("\nCopy this block into Cell 7:\n")
print("RUN_NAMES = [")
for run_name in RUN_NAMES:
    print(f'    "{run_name}",')
print("]\n")
print('EXTRACTION_DIRS = [f"../outputs/extractions/{r}" for r in RUN_NAMES]')

In [ ]:
# Persistent run names for this notebook.
# Paste the RUN_NAMES block printed by Cell 6 here.
RUN_NAMES = [
    "20260617-142701",
    "20260617-142732",
    "20260617-142751",
    "20260617-142809",
    "20260617-142828",
]

EXTRACTION_DIRS = [f"../outputs/extractions/{r}" for r in RUN_NAMES]

if not RUN_NAMES:
    raise RuntimeError("RUN_NAMES is empty. Run Cell 6, then paste its output here.")

print("Using run names:")
for run_name in RUN_NAMES:
    print("  ", run_name)

When the jobs have completed successfully, download the extractions so we can analyze them locally.

In [ ]:
import subprocess

for run_name in RUN_NAMES:
    subprocess.run(
        ["bash", "../scripts/aml_download.sh", "--run-name", run_name],
        check=True,
    )

Make a consensus from the downloaded extractions, and compare with the ground-truth human-transcribed values for each image.

Run the code cell below to build consensus and validation figures.

This produces a consensus in `outputs/consensus_Ciara_data/consensus_1st_order` with transcriptions, summary, and validation figures for all cases.

In [ ]:
import subprocess

cmd = [
    "bash",
    "../scripts/run_consensus_pipeline.sh",
    "--dataset-root",
    "../outputs/consensus_Ciara_data",
    "--variant-name",
    "consensus_1st_order",
    "--threshold",
    "3",  # minimum number of transcriptions required to form a consensus
    "--null-threshold",
    "5", # minimum number of transcriptions required to form a consensus for null
    "--precision",
    "3",
]

for extraction_dir in EXTRACTION_DIRS:
    cmd.extend(["--extraction-dir", extraction_dir])

cmd.extend(
    [
        "--validate",
        "--ground-truth-dir",
        "../test_data/from_Ciara/transcriptions",
        "--validation-sample-denominator",
        "1",
    ]
)

subprocess.run(cmd, check=True)